# Notebook 09: Batch 2 Preprocessing

## Objective

Preprocess the 565 Batch 2 cases that passed the final eligibility
validation in Notebook 08.

The preprocessing pipeline is implemented in `src/preprocessing.py`
and includes:

- CT loading
- segmentation-mask loading
- resampling
- HU clipping
- intensity normalization
- ROI cropping
- padding
- saving processed NumPy arrays
- metadata generation

### Important

`100936_00001` is intentionally excluded from this notebook.

It has a manual-label Z-spacing anomaly and will be investigated
separately after the remaining Batch 2 cases have been processed.

The original raw CT and manual label for that case must remain untouched.

In [1]:
# ============================================================
# NOTEBOOK 09 — BATCH 2 PREPROCESSING
# Project setup
# ============================================================

import sys
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 70)
print("PROJECT SETUP")
print("=" * 70)
print("Project root:")
print(PROJECT_ROOT)

# ------------------------------------------------------------
# Import preprocessing module
# ------------------------------------------------------------

from src import preprocessing as prep

print("\n✓ src.preprocessing imported successfully")

PROJECT SETUP
Project root:
D:\Pancreatic_Cancer_Thesis

✓ src.preprocessing imported successfully


In [2]:
# ============================================================
# DIRECTORIES
# ============================================================

DATA_DIR = PROJECT_ROOT / "data"

RAW_CT_DIR = DATA_DIR / "raw_ct"
LABELS_DIR = DATA_DIR / "labels"

AUTO_LABEL_DIR = LABELS_DIR / "Automatic_Labels"
MANUAL_LABEL_DIR = LABELS_DIR / "Manual_Labels"

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_IMAGES_DIR = PROCESSED_DIR / "images"
PROCESSED_MASKS_DIR = PROCESSED_DIR / "masks"

ELIGIBILITY_FILE = (
    PROCESSED_DIR / "batch2_eligibility.csv"
)

print("=" * 70)
print("DIRECTORY CONFIGURATION")
print("=" * 70)

directories = {
    "Raw CT": RAW_CT_DIR,
    "Automatic labels": AUTO_LABEL_DIR,
    "Manual labels": MANUAL_LABEL_DIR,
    "Processed": PROCESSED_DIR,
    "Processed images": PROCESSED_IMAGES_DIR,
    "Processed masks": PROCESSED_MASKS_DIR,
    "Eligibility": ELIGIBILITY_FILE,
}

for name, path in directories.items():
    print(f"{name:22s}: {path}")
    print(f"{'Exists':22s}: {path.exists()}")
    print()

DIRECTORY CONFIGURATION
Raw CT                : D:\Pancreatic_Cancer_Thesis\data\raw_ct
Exists                : True

Automatic labels      : D:\Pancreatic_Cancer_Thesis\data\labels\Automatic_Labels
Exists                : True

Manual labels         : D:\Pancreatic_Cancer_Thesis\data\labels\Manual_Labels
Exists                : True

Processed             : D:\Pancreatic_Cancer_Thesis\data\processed
Exists                : True

Processed images      : D:\Pancreatic_Cancer_Thesis\data\processed\images
Exists                : True

Processed masks       : D:\Pancreatic_Cancer_Thesis\data\processed\masks
Exists                : True

Eligibility           : D:\Pancreatic_Cancer_Thesis\data\processed\batch2_eligibility.csv
Exists                : True



In [3]:
# ============================================================
# LOAD NOTEBOOK 08 ELIGIBILITY RESULTS
# ============================================================

print("=" * 70)
print("LOADING BATCH 2 ELIGIBILITY RESULTS")
print("=" * 70)

if not ELIGIBILITY_FILE.exists():
    raise FileNotFoundError(
        f"Notebook 08 eligibility file was not found:\n"
        f"{ELIGIBILITY_FILE}"
    )

eligibility = pd.read_csv(ELIGIBILITY_FILE)

print("\nEligibility file:")
print(ELIGIBILITY_FILE)

print("\nShape:")
print(eligibility.shape)

print("\nColumns:")
print(eligibility.columns.tolist())

print("\nFirst 5 rows:")
display(eligibility.head())

LOADING BATCH 2 ELIGIBILITY RESULTS

Eligibility file:
D:\Pancreatic_Cancer_Thesis\data\processed\batch2_eligibility.csv

Shape:
(566, 20)

Columns:
['study_id', 'ct_exists', 'automatic_label_exists', 'manual_label_exists', 'already_processed', 'ready_for_inspection', 'has_any_label', 'has_both_labels', 'has_automatic_only', 'has_manual_only', 'has_no_label', 'ct_readable', 'selected_label_type', 'selected_label_path', 'label_readable', 'geometry_match', 'spacing_match', 'validation_error', 'eligible_for_preprocessing', 'status']

First 5 rows:


,study_id,ct_exists,automatic_label_exists,manual_label_exists,already_processed,ready_for_inspection,has_any_label,has_both_labels,has_automatic_only,has_manual_only,has_no_label,ct_readable,selected_label_type,selected_label_path,label_readable,geometry_match,spacing_match,validation_error,eligible_for_preprocessing,status
0,100546_00001,True,True,False,False,True,True,False,True,False,False,True,automatic,d:\Pancreatic_Cancer_Thesis\data\labels\Automa...,True,True,True,NaN,True,ELIGIBLE
1,100547_00001,True,True,False,False,True,True,False,True,False,False,True,automatic,d:\Pancreatic_Cancer_Thesis\data\labels\Automa...,True,True,True,NaN,True,ELIGIBLE
2,100548_00001,True,True,False,False,True,True,False,True,False,False,True,automatic,d:\Pancreatic_Cancer_Thesis\data\labels\Automa...,True,True,True,NaN,True,ELIGIBLE
3,100549_00001,True,True,False,False,True,True,False,True,False,False,True,automatic,d:\Pancreatic_Cancer_Thesis\data\labels\Automa...,True,True,True,NaN,True,ELIGIBLE
4,100550_00001,True,True,False,False,True,True,False,True,False,False,True,automatic,d:\Pancreatic_Cancer_Thesis\data\labels\Automa...,True,True,True,NaN,True,ELIGIBLE


In [4]:
# ============================================================
# BUILD FINAL BATCH 2 PREPROCESSING LIST
# ============================================================

required_columns = [
    "study_id",
    "selected_label_type",
    "ct_readable",
    "label_readable",
    "geometry_match",
    "spacing_match",
]

missing_columns = [
    col
    for col in required_columns
    if col not in eligibility.columns
]

if missing_columns:
    raise ValueError(
        "Eligibility file is missing required columns:\n"
        + "\n".join(f"  - {col}" for col in missing_columns)
    )

eligible_mask = (
    eligibility["ct_readable"].fillna(False)
    & eligibility["label_readable"].fillna(False)
    & eligibility["geometry_match"].fillna(False)
    & eligibility["spacing_match"].fillna(False)
    & eligibility["selected_label_type"].isin(
        ["automatic", "manual"]
    )
)

batch2_eligible = (
    eligibility.loc[eligible_mask, "study_id"]
    .astype(str)
    .str.strip()
    .sort_values()
    .tolist()
)

# Explicitly exclude the held-out case
HELD_OUT_CASE = "100936_00001"

if HELD_OUT_CASE in batch2_eligible:
    raise RuntimeError(
        f"{HELD_OUT_CASE} unexpectedly appears in the eligible list. "
        "Do not continue until Notebook 08 eligibility is checked."
    )

print("=" * 70)
print("BATCH 2 PREPROCESSING LIST")
print("=" * 70)

print("Total eligibility records :", len(eligibility))
print("Eligible cases            :", len(batch2_eligible))
print("Held-out case             :", HELD_OUT_CASE)

print("\nFirst 20 eligible cases:")
for study_id in batch2_eligible[:20]:
    print(study_id)

BATCH 2 PREPROCESSING LIST
Total eligibility records : 566
Eligible cases            : 565
Held-out case             : 100936_00001

First 20 eligible cases:
100546_00001
100547_00001
100548_00001
100549_00001
100550_00001
100551_00001
100552_00001
100553_00001
100554_00001
100555_00001
100556_00001
100557_00001
100558_00001
100559_00001
100560_00001
100561_00001
100562_00001
100563_00001
100564_00001
100565_00001


In [5]:
# ============================================================
# LABEL TYPE SUMMARY
# ============================================================

eligible_rows = eligibility[
    eligibility["study_id"].astype(str).str.strip().isin(
        batch2_eligible
    )
].copy()

print("=" * 70)
print("SELECTED LABEL SUMMARY")
print("=" * 70)

print(
    eligible_rows["selected_label_type"]
    .value_counts(dropna=False)
)

print("\nExpected:")
print("  automatic : 444")
print("  manual    : 121")

SELECTED LABEL SUMMARY
selected_label_type
automatic    444
manual       121
Name: count, dtype: int64

Expected:
  automatic : 444
  manual    : 121


In [6]:
# ============================================================
# RAW CT FILE CHECK
# ============================================================

print("=" * 70)
print("CHECKING RAW CT FILES")
print("=" * 70)

missing_ct = []

for study_id in batch2_eligible:

    ct_path = RAW_CT_DIR / f"{study_id}_0000.nii.gz"

    if not ct_path.exists():
        missing_ct.append(study_id)

print("Eligible cases :", len(batch2_eligible))
print("Missing CTs    :", len(missing_ct))

if missing_ct:
    print("\nMissing CT files:")
    for study_id in missing_ct:
        print(" ", study_id)

    raise FileNotFoundError(
        f"{len(missing_ct)} eligible CT files are missing."
    )

print("\n✓ All eligible CT files exist.")

CHECKING RAW CT FILES
Eligible cases : 565
Missing CTs    : 0

✓ All eligible CT files exist.


In [7]:
# ============================================================
# LABEL FILE CHECK
# ============================================================

print("=" * 70)
print("CHECKING SELECTED LABEL FILES")
print("=" * 70)

missing_labels = []

for _, row in eligible_rows.iterrows():

    study_id = str(row["study_id"]).strip()
    label_type = row["selected_label_type"]

    if label_type == "automatic":

        label_path = (
            AUTO_LABEL_DIR
            / f"{study_id}.nii.gz"
        )

    elif label_type == "manual":

        label_path = (
            MANUAL_LABEL_DIR
            / f"{study_id}.nii.gz"
        )

    else:
        missing_labels.append(
            (study_id, label_type, "invalid label type")
        )
        continue

    if not label_path.exists():

        missing_labels.append(
            (
                study_id,
                label_type,
                str(label_path),
            )
        )

print("Eligible cases :", len(batch2_eligible))
print("Missing labels :", len(missing_labels))

if missing_labels:

    print("\nProblematic label entries:")

    for item in missing_labels:
        print(item)

    raise FileNotFoundError(
        f"{len(missing_labels)} selected label files are missing."
    )

print("\n✓ All selected label files exist.")

CHECKING SELECTED LABEL FILES


Eligible cases : 565
Missing labels : 0

✓ All selected label files exist.


In [8]:
# ============================================================
# CHECK EXISTING PROCESSED OUTPUTS
# ============================================================

print("=" * 70)
print("CHECKING EXISTING PROCESSED OUTPUTS")
print("=" * 70)

existing_images = []
existing_masks = []

for study_id in batch2_eligible:

    image_path = (
        PROCESSED_IMAGES_DIR
        / f"{study_id}.npy"
    )

    mask_path = (
        PROCESSED_MASKS_DIR
        / f"{study_id}.npy"
    )

    if image_path.exists():
        existing_images.append(study_id)

    if mask_path.exists():
        existing_masks.append(study_id)

print("Existing Batch 2 images :", len(existing_images))
print("Existing Batch 2 masks  :", len(existing_masks))

if existing_images:
    print("\nFirst existing images:")
    for study_id in existing_images[:20]:
        print(" ", study_id)

if existing_masks:
    print("\nFirst existing masks:")
    for study_id in existing_masks[:20]:
        print(" ", study_id)

print("\n✓ Existing outputs will NOT be overwritten.")

CHECKING EXISTING PROCESSED OUTPUTS
Existing Batch 2 images : 565
Existing Batch 2 masks  : 565

First existing images:
  100546_00001
  100547_00001
  100548_00001
  100549_00001
  100550_00001
  100551_00001
  100552_00001
  100553_00001
  100554_00001
  100555_00001
  100556_00001
  100557_00001
  100558_00001
  100559_00001
  100560_00001
  100561_00001
  100562_00001
  100563_00001
  100564_00001
  100565_00001

First existing masks:
  100546_00001
  100547_00001
  100548_00001
  100549_00001
  100550_00001
  100551_00001
  100552_00001
  100553_00001
  100554_00001
  100555_00001
  100556_00001
  100557_00001
  100558_00001
  100559_00001
  100560_00001
  100561_00001
  100562_00001
  100563_00001
  100564_00001
  100565_00001

✓ Existing outputs will NOT be overwritten.


In [9]:
# ============================================================
# BATCH 2 — PROCESS AND SAVE ELIGIBLE CASES
# ============================================================

print("=" * 70)
print("PROCESSING BATCH 2 ELIGIBLE CASES")
print("=" * 70)

print(f"Requested Batch 2 cases: {len(batch2_eligible)}")

# ------------------------------------------------------------
# Process through the project's official production pipeline.
#
# process_dataset() performs:
#   1. load_case()
#   2. preprocess_case()
#   3. save_case()
#   4. clinical metadata merge
#   5. metadata.csv update
# ------------------------------------------------------------

batch2_metadata = prep.process_dataset(
    study_ids=batch2_eligible,
    overwrite=False,
)

print("\n" + "=" * 70)
print("BATCH 2 PROCESSING COMPLETE")
print("=" * 70)

print(f"Metadata rows returned: {len(batch2_metadata)}")

PROCESSING BATCH 2 ELIGIBLE CASES
Requested Batch 2 cases: 565


Processing dataset:   0%|          | 0/565 [00:00<?, ?it/s]

Skipping 100546_00001
Skipping 100547_00001
Skipping 100548_00001
Skipping 100549_00001
Skipping 100550_00001
Skipping 100551_00001
Skipping 100552_00001
Skipping 100553_00001
Skipping 100554_00001
Skipping 100555_00001
Skipping 100556_00001
Skipping 100557_00001
Skipping 100558_00001
Skipping 100559_00001
Skipping 100560_00001
Skipping 100561_00001
Skipping 100562_00001
Skipping 100563_00001
Skipping 100564_00001
Skipping 100565_00001
Skipping 100566_00001
Skipping 100567_00001
Skipping 100568_00001
Skipping 100569_00001
Skipping 100570_00001
Skipping 100571_00001
Skipping 100572_00001
Skipping 100573_00001
Skipping 100574_00001
Skipping 100575_00001
Skipping 100576_00001
Skipping 100577_00001
Skipping 100578_00001
Skipping 100579_00001
Skipping 100580_00001
Skipping 100581_00001
Skipping 100582_00001
Skipping 100583_00001
Skipping 100584_00001
Skipping 100585_00001
Skipping 100586_00001
Skipping 100587_00001
Skipping 100588_00001
Skipping 100589_00001
Skipping 100590_00001
Skipping 1

ValueError: No metadata records were provided.

In [ ]:
# ============================================================
# PREPROCESSING FAILURE REPORT
# ============================================================

print("=" * 70)
print("PREPROCESSING FAILURE REPORT")
print("=" * 70)

failed_df = pd.DataFrame(failed)

print("Failed cases:", len(failed_df))

if len(failed_df) > 0:

    display(failed_df)

    failure_path = (
        PROCESSED_DIR
        / "batch2_preprocessing_failures.csv"
    )

    failed_df.to_csv(
        failure_path,
        index=False,
    )

    print(
        f"\nFailure report saved to:\n"
        f"{failure_path}"
    )

else:

    print("✓ No preprocessing failures.")

PREPROCESSING FAILURE REPORT


NameError: name 'failed' is not defined

In [ ]:
# ============================================================
# VERIFY ACTUAL PROCESSED OUTPUTS
# ============================================================

print("=" * 70)
print("VERIFYING ACTUAL PROCESSED OUTPUT FILES")
print("=" * 70)

image_files = {
    p.stem
    for p in PROCESSED_IMAGES_DIR.glob("*.npy")
}

mask_files = {
    p.stem
    for p in PROCESSED_MASKS_DIR.glob("*.npy")
}

print("Processed image files:", len(image_files))
print("Processed mask files :", len(mask_files))

# ------------------------------------------------------------
# Batch 2 expected cases
# ------------------------------------------------------------

batch2_ids = set(batch2_eligible)

batch2_images = batch2_ids & image_files
batch2_masks = batch2_ids & mask_files

missing_images = batch2_ids - image_files
missing_masks = batch2_ids - mask_files

print("\n" + "-" * 70)
print("BATCH 2 OUTPUT CHECK")
print("-" * 70)

print("Expected Batch 2 cases :", len(batch2_ids))
print("Batch 2 image outputs  :", len(batch2_images))
print("Batch 2 mask outputs   :", len(batch2_masks))
print("Missing images         :", len(missing_images))
print("Missing masks          :", len(missing_masks))

# ------------------------------------------------------------
# Report missing cases
# ------------------------------------------------------------

if missing_images:
    print("\nMissing Batch 2 images:")
    for study_id in sorted(missing_images):
        print(" ", study_id)

if missing_masks:
    print("\nMissing Batch 2 masks:")
    for study_id in sorted(missing_masks):
        print(" ", study_id)

# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("\n" + "=" * 70)

if not missing_images and not missing_masks:
    print("✓ ALL 565 BATCH 2 CASES HAVE IMAGE + MASK OUTPUTS")
else:
    print("⚠ SOME BATCH 2 OUTPUTS ARE MISSING")

print("=" * 70)

VERIFYING ACTUAL PROCESSED OUTPUT FILES
Processed image files: 1122
Processed mask files : 1122


NameError: name 'batch2_eligible' is not defined

In [ ]:
# ============================================================
# SAMPLE PROCESSED CASE INSPECTION — ROBUST VERSION
# ============================================================

print("=" * 70)
print("SAMPLE PROCESSED CASE INSPECTION")
print("=" * 70)

# ------------------------------------------------------------
# Find actual processed files
# ------------------------------------------------------------

actual_image_files = sorted(
    PROCESSED_IMAGES_DIR.glob("*.npy")
)

actual_mask_files = sorted(
    PROCESSED_MASKS_DIR.glob("*.npy")
)

print("Processed image files found :", len(actual_image_files))
print("Processed mask files found  :", len(actual_mask_files))

if len(actual_image_files) == 0:
    raise RuntimeError(
        f"No processed image files found in:\n"
        f"{PROCESSED_IMAGES_DIR}"
    )

if len(actual_mask_files) == 0:
    raise RuntimeError(
        f"No processed mask files found in:\n"
        f"{PROCESSED_MASKS_DIR}"
    )

# ------------------------------------------------------------
# Build matching case IDs from actual files
# ------------------------------------------------------------

actual_image_ids = {
    p.stem: p
    for p in actual_image_files
}

actual_mask_ids = {
    p.stem: p
    for p in actual_mask_files
}

matching_ids = sorted(
    set(actual_image_ids) & set(actual_mask_ids)
)

print("Cases with both outputs      :", len(matching_ids))

if len(matching_ids) == 0:
    raise RuntimeError(
        "No cases have both processed image and mask files."
    )

# ------------------------------------------------------------
# Inspect up to 5 actual cases
# ------------------------------------------------------------

sample_count = min(5, len(matching_ids))

np.random.seed(42)

sample_ids = list(
    np.random.choice(
        matching_ids,
        size=sample_count,
        replace=False,
    )
)

print("\nInspecting cases:")

for study_id in sample_ids:

    image_path = actual_image_ids[study_id]
    mask_path = actual_mask_ids[study_id]

    image = np.load(
        image_path,
        mmap_mode="r",
    )

    mask = np.load(
        mask_path,
        mmap_mode="r",
    )

    print("\n" + "-" * 70)
    print("Study ID:", study_id)

    print("\nImage:")
    print("  Path  :", image_path)
    print("  Shape :", image.shape)
    print("  Dtype :", image.dtype)
    print(
        "  Range :",
        float(image.min()),
        "→",
        float(image.max()),
    )

    print("\nMask:")
    print("  Path  :", mask_path)
    print("  Shape :", mask.shape)
    print("  Dtype :", mask.dtype)
    print("  Labels:", np.unique(mask))

    # --------------------------------------------------------
    # Basic consistency checks
    # --------------------------------------------------------

    if image.shape != mask.shape:
        print("  ⚠ WARNING: image/mask shapes differ!")
    else:
        print("  ✓ Image and mask shapes match.")

print("\n" + "=" * 70)
print("SAMPLE INSPECTION COMPLETE")
print("=" * 70)

SAMPLE PROCESSED CASE INSPECTION
Processed image files found : 1122
Processed mask files found  : 1122
Cases with both outputs      : 1122

Inspecting cases:

----------------------------------------------------------------------
Study ID: 100961_00001

Image:
  Path  : D:\Pancreatic_Cancer_Thesis\data\processed\images\100961_00001.npy
  Shape : (128, 160, 192)
  Dtype : float32
  Range : 0.0 → 1.0

Mask:
  Path  : D:\Pancreatic_Cancer_Thesis\data\processed\masks\100961_00001.npy
  Shape : (128, 160, 192)
  Dtype : uint8
  Labels: [0 2 3 4 5 6]
  ✓ Image and mask shapes match.

----------------------------------------------------------------------
Study ID: 100813_00001

Image:
  Path  : D:\Pancreatic_Cancer_Thesis\data\processed\images\100813_00001.npy
  Shape : (128, 160, 192)
  Dtype : float32
  Range : 0.0 → 1.0

Mask:
  Path  : D:\Pancreatic_Cancer_Thesis\data\processed\masks\100813_00001.npy
  Shape : (128, 160, 192)
  Dtype : uint8
  Labels: [0 2 3 4 5 6]
  ✓ Image and mask shap

In [ ]:
# ============================================================
# BATCH 2 OUTPUT SUMMARY
# ============================================================

print("=" * 70)
print("BATCH 2 OUTPUT SUMMARY")
print("=" * 70)

processed_images = {
    p.stem
    for p in PROCESSED_IMAGES_DIR.glob("*.npy")
}

processed_masks = {
    p.stem
    for p in PROCESSED_MASKS_DIR.glob("*.npy")
}

batch2_set = set(batch2_eligible)

batch2_images = batch2_set & processed_images
batch2_masks = batch2_set & processed_masks

print("Batch 2 eligible cases :", len(batch2_set))
print("Batch 2 image outputs  :", len(batch2_images))
print("Batch 2 mask outputs   :", len(batch2_masks))

print("\nMissing image outputs:")
print(len(batch2_set - processed_images))

print("Missing mask outputs:")
print(len(batch2_set - processed_masks))

if HELD_OUT_CASE in processed_images:
    print(
        "\n⚠ WARNING: held-out case has a processed image."
    )

if HELD_OUT_CASE in processed_masks:
    print(
        "⚠ WARNING: held-out case has a processed mask."
    )

if (
    len(batch2_images) == len(batch2_set)
    and len(batch2_masks) == len(batch2_set)
    and HELD_OUT_CASE not in processed_images
    and HELD_OUT_CASE not in processed_masks
):
    print("\n✓ Batch 2 output files are complete.")
    print("✓ 100936_00001 remains excluded.")

BATCH 2 OUTPUT SUMMARY
Batch 2 eligible cases : 565
Batch 2 image outputs  : 565
Batch 2 mask outputs   : 565

Missing image outputs:
0
Missing mask outputs:
0

✓ Batch 2 output files are complete.
✓ 100936_00001 remains excluded.


In [ ]:
# ============================================================
# DATASET VERIFICATION
# ============================================================

print("=" * 70)
print("RUNNING DATASET VERIFICATION")
print("=" * 70)

summary, report = prep.verify_dataset()

print("\n" + "=" * 70)
print("PROCESSED DATASET VERIFICATION")
print("=" * 70)

print(report)

RUNNING DATASET VERIFICATION


Verifying dataset:   0%|          | 0/1122 [00:00<?, ?it/s]

Processed Dataset Verification
num_cases                : 1122
missing_images           : 0
missing_masks            : 0
invalid_image_shape      : 0
invalid_mask_shape       : 0
invalid_image_dtype      : 0
invalid_mask_dtype       : 0
invalid_image_range      : 0
invalid_mask_labels      : 0
duplicate_study_ids      : 0

✓ All processed cases passed verification.

PROCESSED DATASET VERIFICATION
          study_id status issues  image_min  image_max    mask_labels
0     100000_00001   PASS               0.0        1.0      0,2,3,4,6
1     100001_00001   PASS               0.0        1.0    0,2,3,4,5,6
2     100002_00001   PASS               0.0        1.0  0,1,2,3,4,5,6
3     100003_00001   PASS               0.0        1.0  0,1,2,3,4,5,6
4     100004_00001   PASS               0.0        1.0    0,2,3,4,5,6
...            ...    ...    ...        ...        ...            ...
1117  101108_00001   PASS               0.0        1.0    0,2,3,4,5,6
1118  101109_00001   PASS               

In [ ]:
# ============================================================
# FINAL BATCH 2 PREPROCESSING REPORT
# ============================================================

print("=" * 70)
print("FINAL BATCH 2 PREPROCESSING REPORT")
print("=" * 70)

print(f"Eligible Batch 2 cases       : {len(batch2_eligible)}")
print(f"Successfully processed       : {len(successful)}")
print(f"Skipped existing             : {len(skipped)}")
print(f"Preprocessing failures       : {len(failed)}")

print("-" * 70)

print(f"Batch 2 image outputs        : {len(batch2_images)}")
print(f"Batch 2 mask outputs         : {len(batch2_masks)}")

print("-" * 70)

print(f"Held-out case                : {HELD_OUT_CASE}")
print(
    "Held-out image exists        :",
    (PROCESSED_IMAGES_DIR / f"{HELD_OUT_CASE}.npy").exists()
)
print(
    "Held-out mask exists         :",
    (PROCESSED_MASKS_DIR / f"{HELD_OUT_CASE}.npy").exists()
)

print("=" * 70)

if (
    len(batch2_eligible) == 565
    and len(failed) == 0
    and len(batch2_images) == 565
    and len(batch2_masks) == 565
    and not (
        PROCESSED_IMAGES_DIR / f"{HELD_OUT_CASE}.npy"
    ).exists()
    and not (
        PROCESSED_MASKS_DIR / f"{HELD_OUT_CASE}.npy"
    ).exists()
):

    print("✓ ALL 565 ELIGIBLE BATCH 2 CASES PROCESSED")
    print("✓ 100936_00001 REMAINS ON HOLD")
    print("✓ NO BATCH 2 PREPROCESSING FAILURES")
    
else:

    print("⚠ BATCH 2 REQUIRES REVIEW")

FINAL BATCH 2 PREPROCESSING REPORT
Eligible Batch 2 cases       : 565
Successfully processed       : 0
Skipped existing             : 0
Preprocessing failures       : 0
----------------------------------------------------------------------
Batch 2 image outputs        : 0
Batch 2 mask outputs         : 0
----------------------------------------------------------------------
Held-out case                : 100936_00001
Held-out image exists        : False
Held-out mask exists         : False
⚠ BATCH 2 REQUIRES REVIEW


In [ ]:
# ============================================================
# INSPECT PREPROCESS_CASE IMPLEMENTATION
# ============================================================

import inspect

print("=" * 70)
print("preprocess_case() SOURCE")
print("=" * 70)

print(inspect.getsource(prep.preprocess_case))

preprocess_case() SOURCE
def preprocess_case(
    case,
    target_spacing=DEFAULT_TARGET_SPACING,
    crop_size=DEFAULT_ROI_SIZE,
    hu_window=DEFAULT_HU_WINDOW,
):
    """
    Complete preprocessing pipeline for one PANORAMA case.

    Processing order
    ----------------
    1. Resample CT and mask
    2. Clip CT Hounsfield Units
    3. Normalize CT intensities
    4. Crop fixed pancreas ROI
    5. Pad to fixed output size

    Parameters
    ----------
    case : dict
        Output of ``load_case()``.

    target_spacing : tuple
        Target voxel spacing (z, y, x).

    crop_size : tuple
        Desired ROI size (depth, height, width).

    hu_window : tuple
        (lower, upper) HU clipping window.

    Returns
    -------
    dict
        Fully preprocessed case.
    """

    new_case = case.copy()

    # ---------------------------------------------------------
    # Step 1: Resample
    # ---------------------------------------------------------

    new_case = resample_

In [ ]:
# ============================================================
# FIND FUNCTIONS THAT SAVE PROCESSED DATA
# ============================================================

import inspect

print("=" * 70)
print("POTENTIAL DATASET / SAVE FUNCTIONS")
print("=" * 70)

for name in dir(prep):
    if name.startswith("_"):
        continue

    obj = getattr(prep, name)

    if callable(obj):
        try:
            source = inspect.getsource(obj)

            # Look for functions that actually write files,
            # create metadata, or process a full dataset.
            keywords = [
                "np.save",
                "to_csv",
                "metadata",
                "PROCESSED_IMAGES_DIR",
                "PROCESSED_MASKS_DIR",
                "save",
                "dataset",
            ]

            if any(keyword in source for keyword in keywords):
                print("\n" + "=" * 70)
                print(f"FUNCTION: {name}")
                print("=" * 70)
                print(source)

        except (OSError, TypeError):
            pass

POTENTIAL DATASET / SAVE FUNCTIONS

FUNCTION: analyze_roi_sizes
def analyze_roi_sizes(
    label=4,
    target_spacing=DEFAULT_TARGET_SPACING,
    progress_every=25,
):
    """
    Analyze bounding-box sizes of a structure across the dataset.

    Parameters
    ----------
    label : int
        Structure label.
        4 = pancreas
        1 = tumor

    target_spacing : tuple
        Spacing used before measuring the ROI.

    progress_every : int
        Print progress every N cases.

    Returns
    -------
    pandas.DataFrame
        Columns:
        study_id
        depth
        height
        width
        volume
    """

    results = []

    study_ids = list_ct_cases()

    total = len(study_ids)

    for i, study_id in enumerate(study_ids, start=1):

        if i % progress_every == 0 or i == total:
            print(f"{i}/{total}")

        try:

            mask_image, _, _, _ = load_mask(study_id)

            mask_image = resample_mask(
                mask_image,
    

In [10]:
image_files = {
    p.stem
    for p in PROCESSED_IMAGES_DIR.glob("*.npy")
}

mask_files = {
    p.stem
    for p in PROCESSED_MASKS_DIR.glob("*.npy")
}

batch2_ids = set(batch2_eligible)

print("Total processed images:", len(image_files))
print("Total processed masks :", len(mask_files))
print("Batch 2 images        :", len(batch2_ids & image_files))
print("Batch 2 masks         :", len(batch2_ids & mask_files))

Total processed images: 1122
Total processed masks : 1122
Batch 2 images        : 565
Batch 2 masks         : 565


In [11]:
# ============================================================
# FINAL PROCESSED DATASET METADATA CHECK
# ============================================================

print("=" * 70)
print("FINAL PROCESSED DATASET METADATA CHECK")
print("=" * 70)

metadata_path = PROCESSED_DIR / "metadata.csv"

print("Metadata path:", metadata_path)
print("Exists:", metadata_path.exists())

if metadata_path.exists():
    metadata = pd.read_csv(metadata_path)

    print("\nMetadata rows:", len(metadata))
    print("Metadata columns:")
    print(metadata.columns.tolist())

    print("\n" + "-" * 70)

    processed_image_ids = {
        p.stem for p in PROCESSED_IMAGES_DIR.glob("*.npy")
    }

    processed_mask_ids = {
        p.stem for p in PROCESSED_MASKS_DIR.glob("*.npy")
    }

    metadata_ids = set(metadata["study_id"].astype(str))

    print("Processed image cases :", len(processed_image_ids))
    print("Processed mask cases  :", len(processed_mask_ids))
    print("Metadata cases        :", len(metadata_ids))

    missing_from_metadata = (
        processed_image_ids - metadata_ids
    )

    metadata_without_image = (
        metadata_ids - processed_image_ids
    )

    metadata_without_mask = (
        metadata_ids - processed_mask_ids
    )

    print("\nCases with processed image but no metadata:",
          len(missing_from_metadata))

    print("Metadata cases without image:",
          len(metadata_without_image))

    print("Metadata cases without mask:",
          len(metadata_without_mask))

    if (
        len(metadata_ids) == 1122
        and not missing_from_metadata
        and not metadata_without_image
        and not metadata_without_mask
    ):
        print("\n✓ METADATA AND PROCESSED FILES ARE CONSISTENT")
    else:
        print("\n⚠ Metadata needs reconciliation.")
else:
    print("\n⚠ metadata.csv does not exist.")

FINAL PROCESSED DATASET METADATA CHECK
Metadata path: D:\Pancreatic_Cancer_Thesis\data\processed\metadata.csv
Exists: True

Metadata rows: 1122
Metadata columns:
['study_id', 'label', 'image_path', 'mask_path', 'image_shape', 'mask_shape', 'image_dtype', 'mask_dtype', 'mask_labels', 'patient_id', 'patient_age', 'patient_sex', 'scanner', 'diagnosis', 'diagnosis_source', 'roi_size', 'target_spacing', 'hu_window']

----------------------------------------------------------------------
Processed image cases : 1122
Processed mask cases  : 1122
Metadata cases        : 1122

Cases with processed image but no metadata: 0
Metadata cases without image: 0
Metadata cases without mask: 0

✓ METADATA AND PROCESSED FILES ARE CONSISTENT


In [3]:
# ============================================================
# PROCESS 100936_00001 USING THE REPAIRED MANUAL LABEL
# ============================================================
#
# IMPORTANT:
# - Original PANORAMA manual label is NOT modified.
# - Original CT is NOT modified.
# - Uses the same preprocessing/save/metadata functions as
#   prep.process_dataset().
# - Only the mask source is replaced with the repaired copy.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

study_id = "100936_00001"

PROJECT_ROOT = Path.cwd().parent

ORIGINAL_CT_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw_ct"
    / f"{study_id}_0000.nii.gz"
)

ORIGINAL_LABEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "labels"
    / "Manual_Labels"
    / f"{study_id}.nii.gz"
)

REPAIRED_LABEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "anomaly_investigation"
    / f"{study_id}_repaired.nii.gz"
)

PROCESSED_IMAGE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "images"
    / f"{study_id}.npy"
)

PROCESSED_MASK_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "masks"
    / f"{study_id}.npy"
)

METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "metadata.csv"
)


print("=" * 70)
print("PROCESSING REPAIRED CASE: 100936_00001")
print("=" * 70)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

for path, description in [
    (ORIGINAL_CT_PATH, "Original CT"),
    (ORIGINAL_LABEL_PATH, "Original PANORAMA label"),
    (REPAIRED_LABEL_PATH, "Repaired label"),
]:
    if not path.exists():
        raise FileNotFoundError(
            f"{description} not found:\n{path}"
        )

print("\n✓ Original CT exists")
print("✓ Original PANORAMA label exists")
print("✓ Repaired label exists")


# ------------------------------------------------------------
# Do not overwrite an existing processed case accidentally
# ------------------------------------------------------------

if PROCESSED_IMAGE_PATH.exists():
    raise FileExistsError(
        f"Processed image already exists:\n"
        f"{PROCESSED_IMAGE_PATH}\n\n"
        "Stop here rather than overwriting it."
    )

if PROCESSED_MASK_PATH.exists():
    raise FileExistsError(
        f"Processed mask already exists:\n"
        f"{PROCESSED_MASK_PATH}\n\n"
        "Stop here rather than overwriting it."
    )


# ------------------------------------------------------------
# 1. Load the case through the existing project loader
#
# This loads the original CT and original manual label.
# We will immediately replace the label fields below.
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("STEP 1 — LOADING CASE")
print("-" * 70)

case = prep.load_case(study_id)

print("Case loaded.")
print("Case keys:")
print(list(case.keys()))


# ------------------------------------------------------------
# 2. Replace ONLY the mask with the repaired NIfTI
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("STEP 2 — INJECTING REPAIRED LABEL")
print("-" * 70)

repaired_mask_image = sitk.ReadImage(
    str(REPAIRED_LABEL_PATH)
)

repaired_mask_array = sitk.GetArrayFromImage(
    repaired_mask_image
).astype(np.uint8)


print("Repaired mask image:")
print("  Size    :", repaired_mask_image.GetSize())
print("  Spacing :", repaired_mask_image.GetSpacing())
print("  Origin  :", repaired_mask_image.GetOrigin())
print("  Labels  :", np.unique(repaired_mask_array))


# Replace the mask fields in the case.
#
# These are the fields load_case() provides and the
# preprocessing functions use.
case["mask_image"] = repaired_mask_image
case["mask_array"] = repaired_mask_array
case["mask_type"] = "manual_repaired"
case["mask_path"] = str(REPAIRED_LABEL_PATH)


# ------------------------------------------------------------
# Verify CT ↔ repaired mask geometry before preprocessing
# ------------------------------------------------------------

print("\nCT geometry:")
print("  Size    :", case["ct_image"].GetSize())
print("  Spacing :", case["ct_image"].GetSpacing())
print("  Origin  :", case["ct_image"].GetOrigin())

print("\nRepaired mask geometry:")
print("  Size    :", case["mask_image"].GetSize())
print("  Spacing :", case["mask_image"].GetSpacing())
print("  Origin  :", case["mask_image"].GetOrigin())

if case["ct_image"].GetSize() != case["mask_image"].GetSize():
    raise ValueError(
        "CT and repaired mask sizes do not match."
    )

if not np.allclose(
    case["ct_image"].GetSpacing(),
    case["mask_image"].GetSpacing(),
    atol=1e-6,
):
    raise ValueError(
        "CT and repaired mask spacing do not match."
    )

if not np.allclose(
    case["ct_image"].GetOrigin(),
    case["mask_image"].GetOrigin(),
    atol=1e-5,
):
    raise ValueError(
        "CT and repaired mask origins do not match."
    )

if not np.allclose(
    case["ct_image"].GetDirection(),
    case["mask_image"].GetDirection(),
    atol=1e-6,
):
    raise ValueError(
        "CT and repaired mask directions do not match."
    )

print("\n✓ CT and repaired mask geometry match.")


# ------------------------------------------------------------
# 3. Run EXACT project preprocessing function
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("STEP 3 — RUNNING STANDARD PREPROCESSING")
print("-" * 70)

processed_case = prep.preprocess_case(case)


# ------------------------------------------------------------
# 4. Save using the project's standard save_case()
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("STEP 4 — SAVING PROCESSED CASE")
print("-" * 70)

record = prep.save_case(
    processed_case
)

print("\n✓ Processed arrays saved.")

print("Image:")
print(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "images"
    / f"{study_id}.npy"
)

print("Mask:")
print(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "masks"
    / f"{study_id}.npy"
)


# ------------------------------------------------------------
# 5. Merge clinical metadata exactly as process_dataset()
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("STEP 5 — UPDATING METADATA")
print("-" * 70)

clinical = prep.load_metadata().copy()

clinical = clinical.rename(
    columns={
        "PANORAMA_study_id": "study_id"
    }
)

clinical = clinical.set_index(
    "study_id"
)


if study_id in clinical.index:

    record = prep.merge_clinical_metadata(
        record,
        clinical.loc[study_id]
    )

    print("✓ Clinical metadata merged.")

else:

    print(
        f"⚠ Clinical metadata not found for {study_id}"
    )


# ------------------------------------------------------------
# 6. Update metadata.csv using the standard function
# ------------------------------------------------------------

updated_metadata = prep.update_metadata(
    [record],
    metadata_path=METADATA_PATH,
)

print(
    "\n✓ metadata.csv updated."
)

print(
    "Metadata rows:",
    len(updated_metadata)
)


# ============================================================
# FINAL CASE CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL 100936_00001 PROCESSING CHECK")
print("=" * 70)

print(
    "Processed image exists:",
    PROCESSED_IMAGE_PATH.exists()
)

print(
    "Processed mask exists :",
    PROCESSED_MASK_PATH.exists()
)

print(
    "Metadata contains case :",
    updated_metadata["study_id"]
    .astype(str)
    .eq(study_id)
    .any()
)


# ------------------------------------------------------------
# Load saved arrays and inspect
# ------------------------------------------------------------

processed_image = np.load(
    PROCESSED_IMAGE_PATH,
    mmap_mode="r",
)

processed_mask = np.load(
    PROCESSED_MASK_PATH,
    mmap_mode="r",
)


print("\nImage:")
print("  Shape :", processed_image.shape)
print("  Dtype :", processed_image.dtype)
print(
    "  Range :",
    float(processed_image.min()),
    "→",
    float(processed_image.max()),
)

print("\nMask:")
print("  Shape :", processed_mask.shape)
print("  Dtype :", processed_mask.dtype)
print(
    "  Labels :",
    np.unique(processed_mask)
)


# ------------------------------------------------------------
# Basic assertions
# ------------------------------------------------------------

assert processed_image.shape == (
    128,
    160,
    192,
)

assert processed_mask.shape == (
    128,
    160,
    192,
)

assert processed_image.dtype == np.float32
assert processed_mask.dtype == np.uint8

assert (
    float(processed_image.min()) >= 0.0
)

assert (
    float(processed_image.max()) <= 1.0
)

assert set(
    np.unique(processed_mask).astype(int)
).issubset(
    {0, 1, 2, 3, 4, 5, 6}
)

print("\n✓ Processed image passed basic validation.")
print("✓ Processed mask passed basic validation.")

print("\n" + "=" * 70)
print("✓ 100936_00001 SUCCESSFULLY PROCESSED")
print("=" * 70)

print("\nImportant:")
print("Original PANORAMA manual label was NOT modified.")
print("The repaired label remains in:")
print(REPAIRED_LABEL_PATH)

PROCESSING REPAIRED CASE: 100936_00001

✓ Original CT exists
✓ Original PANORAMA label exists
✓ Repaired label exists

----------------------------------------------------------------------
STEP 1 — LOADING CASE
----------------------------------------------------------------------
Case loaded.
Case keys:
['study_id', 'ct_path', 'mask_path', 'mask_type', 'ct_image', 'mask_image', 'ct_array', 'mask_array', 'spacing', 'origin', 'direction']

----------------------------------------------------------------------
STEP 2 — INJECTING REPAIRED LABEL
----------------------------------------------------------------------
Repaired mask image:
  Size    : (1024, 1024, 406)
  Spacing : (0.3910059928894043, 0.3910059928894043, 0.7543209791183472)
  Origin  : (-186.32794189453125, -199.99981689453125, 1500.5)
  Labels  : [0 1 2 3 4 5 6]

CT geometry:
  Size    : (1024, 1024, 406)
  Spacing : (0.3910059928894043, 0.3910059928894043, 0.7543209791183472)
  Origin  : (-186.32794189453125, -199.999816894

In [4]:
# ============================================================
# FINAL FULL DATASET VERIFICATION
# ============================================================

print("=" * 70)
print("FINAL FULL DATASET VERIFICATION")
print("=" * 70)

summary, report = prep.verify_dataset()

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

for key, value in summary.items():
    print(f"{key:<25}: {value}")

FINAL FULL DATASET VERIFICATION


Verifying dataset:   0%|          | 0/1123 [00:00<?, ?it/s]

Processed Dataset Verification
num_cases                : 1123
missing_images           : 0
missing_masks            : 0
invalid_image_shape      : 0
invalid_mask_shape       : 0
invalid_image_dtype      : 0
invalid_mask_dtype       : 0
invalid_image_range      : 0
invalid_mask_labels      : 0
duplicate_study_ids      : 0

✓ All processed cases passed verification.

FINAL SUMMARY
num_cases                : 1123
missing_images           : 0
missing_masks            : 0
invalid_image_shape      : 0
invalid_mask_shape       : 0
invalid_image_dtype      : 0
invalid_mask_dtype       : 0
invalid_image_range      : 0
invalid_mask_labels      : 0
duplicate_study_ids      : 0
